# EDA — Exploratory Data Analysis
## Dataset: Fruit and Vegetable Image Recognition (Kaggle)

Notebook này thực hiện phân tích khám phá dữ liệu trước khi huấn luyện model, theo đúng yêu cầu môn học:

- Tổng số ảnh, tổng số class, tên từng class
- Số lượng ảnh của từng class (train/validation/test) và tỉ lệ chia tập
- Phân bố dữ liệu giữa các class (class imbalance)
- Kích thước ảnh, định dạng ảnh, ảnh lỗi/bất thường
- Ảnh mẫu của từng class (lưới 6×6)
- **Rút trích đặc trưng:** minh họa HOG, histogram màu
- **Giảm chiều PCA:** số chiều trước/sau, % phương sai giữ lại
- **Tương quan giữa các đặc trưng**

**Yêu cầu trước khi chạy notebook này:** dataset đã được tải về `data/raw/` theo cấu trúc `train/`, `validation/`, `test/` — xem `TRAINING_GUIDE.md` hoặc `scripts/download_dataset.py`.

> Mọi biểu đồ được lưu tự động vào `reports/figures/` để đưa vào báo cáo Word. Các dòng **"Nhận xét tự động"** được tính từ số liệu thật khi chạy; các ô **"Nhận xét"** dạng markdown cần người xem biểu đồ rồi điền thêm.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from configs.config import CONFIG, ensure_dirs
from src.data.dataset import discover_dataset
from src.data.eda import (
    class_distribution_df,
    plot_class_distribution,
    detect_class_imbalance,
    sample_images_grid,
    image_size_stats,
    plot_image_size_distribution,
)
from src.features.hog_features import batch_extract_features, get_feature_dim, COLOR_HIST_BINS, COLOR_WEIGHT
from src.features.pca_wrapper import PCAWrapper
from src.features.feature_eda import (
    sample_paths_balanced,
    plot_hog_examples,
    plot_color_histograms,
    plot_dimension_reduction,
    plot_color_correlation,
    correlation_matrix,
    plot_pca_scatter,
)

%matplotlib inline

ensure_dirs()
FIG_DIR = CONFIG["paths"]["figures_dir"]          # biểu đồ lưu ở đây cho báo cáo
def fig_path(name):
    return os.path.join(FIG_DIR, name)

## 1. Khám phá cấu trúc dataset thực tế trên đĩa

Không giả định trước số class/số ảnh — đọc trực tiếp từ thư mục `data/raw/`.

In [ ]:
splits, info = discover_dataset(CONFIG)

print(f"Tổng số class: {info.num_classes}")
print(f"Tổng số ảnh (train+val+test): {info.total_images}")
print(f"\nTên các class:\n{info.class_names}")

In [ ]:
for split, counts in info.counts_per_split.items():
    print(f"{split}: {sum(counts.values())} ảnh, {len(counts)} class")

### 1.1 Tỉ lệ chia train / validation / test

In [ ]:
split_totals = pd.Series({s: sum(c.values()) for s, c in info.counts_per_split.items()})
split_pct = split_totals / split_totals.sum() * 100

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(split_totals.index, split_totals.values, color=["#4C9A6B", "#E07B39", "#5B8DB8"][: len(split_totals)])
for bar, n, p in zip(bars, split_totals.values, split_pct.values):
    ax.text(bar.get_x() + bar.get_width() / 2, n, f"{n}\n({p:.1f}%)", ha="center", va="bottom")
ax.set_title("Số ảnh theo từng tập (train / validation / test)")
ax.set_ylabel("Số ảnh")
ax.set_ylim(top=split_totals.max() * 1.18)
fig.tight_layout()
fig.savefig(fig_path("split_distribution.png"), dpi=150)
plt.show()

print("Nhận xét tự động:")
print("- Tỉ lệ chia: " + " / ".join(f"{s} {p:.1f}%" for s, p in split_pct.items()) + ".")
print("- Nếu dùng bộ train/validation/test tách sẵn của Kaggle (tách theo ảnh), cần kiểm tra thêm không có ảnh trùng giữa các tập để tránh rò rỉ dữ liệu (data leakage).")

## 2. Class distribution — số lượng ảnh mỗi class

In [ ]:
df = pd.DataFrame(class_distribution_df(info))
pivot = df.pivot(index="class", columns="split", values="count").fillna(0).astype(int)
pivot

In [ ]:
fig = plot_class_distribution(info, split="train", save_path=fig_path("class_distribution_train.png"))
plt.show()

## 3. Class imbalance

In [ ]:
imbalance = detect_class_imbalance(info, split="train")
imbalance

In [ ]:
r = imbalance
print("Nhận xét tự động:")
print(f"- Class nhiều ảnh nhất: '{r['max_class']}' ({r['max_count']} ảnh); ít nhất: '{r['min_class']}' ({r['min_count']} ảnh); trung bình {r['mean_count']:.1f} ảnh/class.")
if r["is_imbalanced"]:
    print(f"- Độ chênh (max-min)/mean = {r['relative_range_ratio']:.2f} vượt ngưỡng 0.3 → dataset MẤT CÂN BẰNG; cần class_weight và dùng macro F1 thay vì chỉ Accuracy.")
else:
    print(f"- Độ chênh (max-min)/mean = {r['relative_range_ratio']:.2f} không vượt ngưỡng 0.3 → dataset tương đối CÂN BẰNG; Accuracy là thước đo đáng tin cậy.")

**Nhận xét:** nếu `is_imbalanced=True`, cần cân nhắc: dùng `class_weight` khi train, hoặc oversampling/augmentation mạnh hơn cho class ít ảnh, và **không đánh giá model chỉ bằng Accuracy** (macro F1 sẽ phản ánh đúng hơn khi có imbalance).

## 4. Ảnh mẫu của từng class (lưới 6×6)

In [ ]:
fig = sample_images_grid(splits["train"], info.class_names, save_path=fig_path("sample_images.png"))
plt.show()

**Nhận xét:** *(TN 2 điền sau khi xem lưới ảnh — gợi ý: nền ảnh sạch hay phức tạp? Có class nào nhìn giống nhau, dễ nhầm — vd `bell pepper` / `capsicum` / `paprika`, `corn` / `sweetcorn`? Có ảnh sai nhãn không?)*

## 5. Kích thước & định dạng ảnh

In [ ]:
sizes, corrupted = image_size_stats(splits["train"])
print(f"Số ảnh đã kiểm tra (mẫu): {len(sizes)}")
print(f"Số kích thước khác nhau: {len(set(sizes))}")
print(f"Số ảnh lỗi/không đọc được (trong mẫu): {len(corrupted)}")
if corrupted:
    print("Danh sách ảnh lỗi:", corrupted)

In [ ]:
fig = plot_image_size_distribution(sizes, save_path=fig_path("image_size_distribution.png"))
plt.show()

In [ ]:
w = np.array([s[0] for s in sizes]); h = np.array([s[1] for s in sizes])
ext_counts = Counter(Path(f).suffix.lower() for files in splits["train"].values() for f in files)
near_square = np.mean(np.isclose(w / h, 1.0, atol=0.05)) * 100

print("Nhận xét tự động:")
print(f"- Chiều rộng: nhỏ nhất {w.min()}px, trung vị {int(np.median(w))}px, lớn nhất {w.max()}px.")
print(f"- Chiều cao : nhỏ nhất {h.min()}px, trung vị {int(np.median(h))}px, lớn nhất {h.max()}px.")
print(f"- {near_square:.0f}% ảnh có tỉ lệ gần vuông; có {len(set(sizes))} kích thước khác nhau trong {len(sizes)} ảnh mẫu.")
if len(set(sizes)) > 1:
    print("→ Ảnh KHÔNG đồng đều kích thước: bắt buộc resize về kích thước cố định (HOG/SVM: 128×128; CNN: 224×224).")
else:
    print("→ Ảnh đồng đều kích thước.")
print(f"- Định dạng file (tập train): {dict(ext_counts)}")

## 6. Rút trích đặc trưng HOG + Color Histogram

Trích đặc trưng trên một **mẫu cân bằng** (`N_PER_CLASS` ảnh/class, lấy từ tập train, cố định seed) để các biểu đồ ở dưới chạy nhanh nhưng vẫn đại diện cho mọi class. Tăng `N_PER_CLASS` để đường cong PCA chính xác hơn (đổi lại chạy lâu hơn).

Đây cũng chính là pipeline đặc trưng mà SVM sẽ dùng: `ảnh → resize 128×128 → HOG (8100 số) + histogram màu RGB (96 số) → vector 8196 số`.

In [ ]:
N_PER_CLASS = 30

paths, labels = sample_paths_balanced(splits["train"], n_per_class=N_PER_CLASS)
dims = get_feature_dim()
print(f"Mẫu EDA: {len(paths)} ảnh ({N_PER_CLASS} ảnh/class × {info.num_classes} class)")
print(f"Số chiều đặc trưng: HOG {dims['hog']} + màu {dims['color']} = {dims['total']}")

# on_error="skip": ảnh hỏng bị bỏ qua (không làm dừng notebook); lọc paths/labels theo mask để không lệch nhãn
X, ok = batch_extract_features(paths, on_error="skip", return_valid_mask=True)
if not ok.all():
    print(f"⚠ Bỏ qua {int((~ok).sum())} ảnh hỏng:", [p for p, k in zip(paths, ok) if not k])
paths = [p for p, k in zip(paths, ok) if k]
labels = [l for l, k in zip(labels, ok) if k]
print(f"Ma trận đặc trưng: {X.shape}  (dtype {X.dtype})")

### 6.1 Minh họa HOG — ảnh gốc và ảnh HOG

In [ ]:
fig = plot_hog_examples(splits["train"], info.class_names, n_examples=3,
                        save_path=fig_path("hog_visualization.png"))
plt.show()

**Nhận xét:** HOG chỉ giữ lại thông tin về **đường viền và hướng cạnh** (các nét sáng trong ảnh HOG), bỏ hoàn toàn màu sắc. Vì vậy hình dạng đặc trưng (quả tròn, củ dài, bề mặt sần...) vẫn thấy rõ, nhưng hai loại có hình dạng giống nhau và khác màu sẽ khó phân biệt nếu chỉ dùng HOG — đó là lý do ghép thêm histogram màu.

### 6.2 Histogram màu của một số class

In [ ]:
fig = plot_color_histograms(splits["train"], info.class_names, n_examples=3,
                            save_path=fig_path("color_histograms.png"))
plt.show()

**Nhận xét:** *(điền sau khi xem — gợi ý: các class khác màu cho đường histogram khác nhau rõ rệt ở kênh nào? Ảnh nền trắng/sáng tạo ra đỉnh nhọn ở bin cuối của cả 3 kênh.)*

### 6.3 Đóng góp của HOG và màu vào tổng phương sai

PCA giữ lại những hướng có **phương sai lớn nhất**. HOG có 8100 chiều đã chuẩn hoá theo block, còn histogram màu chỉ là tỉ lệ pixel nên phương sai rất nhỏ. Nếu ghép thẳng, PCA sẽ gần như bỏ qua màu — dù màu là đặc trưng rất mạnh để phân biệt rau củ quả. Vì vậy phần màu được nhân với `COLOR_WEIGHT` trước khi ghép.

In [ ]:
n_hog = dims["hog"]
var_hog = X[:, :n_hog].var(axis=0).sum()
var_color_weighted = X[:, n_hog:].var(axis=0).sum()          # X đã nhân COLOR_WEIGHT
var_color_raw = var_color_weighted / COLOR_WEIGHT ** 2        # phương sai nếu KHÔNG nhân trọng số

share_raw = 100 * var_color_raw / (var_hog + var_color_raw)
share_weighted = 100 * var_color_weighted / (var_hog + var_color_weighted)

fig, ax = plt.subplots(figsize=(6, 4))
labels_bar = ["Màu — không trọng số\n(weight = 1)", f"Màu — có trọng số\n(weight = {COLOR_WEIGHT:g})"]
vals = [share_raw, share_weighted]
bars = ax.bar(labels_bar, vals, color=["#BBBBBB", "#E07B39"])
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.1f}%", ha="center", va="bottom")
ax.set_ylabel("% tổng phương sai đặc trưng")
ax.set_title("Phần đóng góp của đặc trưng màu vào tổng phương sai")
ax.set_ylim(top=max(vals) * 1.2)
fig.tight_layout()
fig.savefig(fig_path("color_variance_share.png"), dpi=150)
plt.show()

print("Nhận xét tự động:")
print(f"- Màu chỉ có {dims['color']} chiều so với {n_hog} chiều HOG và nếu không nhân trọng số chỉ đóng góp {share_raw:.2f}% tổng phương sai → PCA gần như bỏ qua màu.")
print(f"- Sau khi nhân COLOR_WEIGHT = {COLOR_WEIGHT:g}, màu đóng góp {share_weighted:.1f}% → được PCA giữ lại hợp lý hơn.")

## 7. Giảm chiều bằng PCA

Fit PCA với **toàn bộ** số component tối đa có thể để vẽ được đường cong phương sai tích lũy đầy đủ, từ đó chọn số chiều `n_components` hợp lý cho SVM.

> Lưu ý: ở đây PCA chỉ fit trên mẫu EDA (tập train) để *phân tích*. Khi huấn luyện SVM thật, PCA phải fit **chỉ trên tập train** rồi `transform` val/test — tuyệt đối không fit trên val/test (data leakage).

In [ ]:
n_max = min(X.shape)
pca_full = PCAWrapper(n_components=n_max).fit(X)

cum = np.cumsum(pca_full.get_explained_variance_ratio()) * 100
k_default = min(200, n_max)                       # số chiều mặc định dự kiến cho SVM
ks = {f"Giữ {int(t*100)}%": pca_full.n_components_for_variance(t) for t in (0.90, 0.95, 0.99)}

print(f"Số chiều ban đầu: {X.shape[1]}")
for name, k in ks.items():
    print(f"{name} phương sai cần {k} component")
print(f"Với {k_default} component: giữ {cum[k_default - 1]:.1f}% phương sai")

In [ ]:
fig = pca_full.plot_explained_variance(save_path=fig_path("pca_explained_variance.png"))
plt.show()

In [ ]:
variance_table = {name: (k, cum[k - 1]) for name, k in ks.items() if k is not None}
variance_table[f"{k_default} component\n(mặc định)"] = (k_default, cum[k_default - 1])

fig = plot_dimension_reduction(X.shape[1], variance_table, save_path=fig_path("pca_dimensions_before_after.png"))
plt.show()

k95 = ks["Giữ 95%"]
print("Nhận xét tự động:")
print(f"- PCA nén vector từ {X.shape[1]} chiều xuống còn {k95} chiều (giảm {100 * (1 - k95 / X.shape[1]):.1f}%) mà vẫn giữ 95% phương sai.")
print(f"- Chọn {k_default} component: giữ {cum[k_default - 1]:.1f}% phương sai — SVM huấn luyện trên vector ngắn hơn ~{X.shape[1] / k_default:.0f} lần.")
print(f"- (Ước lượng trên mẫu {X.shape[0]} ảnh; số component tối đa bị chặn bởi số ảnh. Dùng toàn bộ tập train sẽ cho con số chính xác hơn.)")

## 8. Tương quan giữa các đặc trưng

### 8.1 Heatmap tương quan của các đặc trưng màu (96 số)

In [ ]:
X_color = X[:, dims["hog"]:]                       # 96 cột cuối = histogram màu
fig = plot_color_correlation(X_color, save_path=fig_path("color_feature_correlation.png"))
plt.show()

corr = correlation_matrix(X_color)
iu = np.triu_indices_from(corr, k=1)
pairs = corr[iu]
n_high = int((np.abs(pairs) > 0.8).sum())
print("Nhận xét tự động:")
print(f"- Tương quan tuyệt đối trung bình giữa các cặp đặc trưng màu: {np.abs(pairs).mean():.2f}.")
pct_high = 100 * n_high / len(pairs)
print(f"- Có {n_high}/{len(pairs)} cặp ({pct_high:.1f}%) có |r| > 0.8.")
if pct_high >= 5:
    print("→ Nhiều cặp đặc trưng màu tương quan mạnh (bin màu gần nhau cùng tăng/giảm) → thông tin dư thừa, là một lý do nên dùng PCA để nén trước khi đưa vào SVM.")
else:
    print("→ Đa số các cặp đặc trưng màu ít tương quan trực tiếp; tuy vậy 8100 đặc trưng HOG lân cận nhau vẫn có tương quan không gian cao, nên PCA vẫn hữu ích để nén.")

### 8.2 Phân bố các class trên 2 thành phần chính đầu tiên

In [ ]:
Z = pca_full.transform(X)[:, :2]
classes_shown = info.class_names[:8]
fig = plot_pca_scatter(Z, labels, classes_to_show=classes_shown, save_path=fig_path("pca_scatter.png"))
plt.show()

**Nhận xét:** *(điền sau khi xem — gợi ý: class nào tách riêng khỏi nhóm, class nào chồng lên nhau? Chỉ 2 thành phần đầu nên chồng lấn nhiều là bình thường; SVM dùng toàn bộ các chiều sau PCA nên phân biệt tốt hơn nhiều so với hình này.)*

## 9. Kết luận EDA

Bảng số liệu tổng hợp (tự tính từ các cell phía trên):

In [ ]:
summary = pd.Series({
    "Tổng số class": info.num_classes,
    "Tổng số ảnh (train+val+test)": info.total_images,
    "Tỉ lệ train/val/test (%)": " / ".join(f"{p:.1f}" for p in split_pct.values),
    "Class imbalance": "Có" if imbalance["is_imbalanced"] else "Không (tương đối cân bằng)",
    "Số kích thước ảnh khác nhau (mẫu)": len(set(sizes)),
    "Ảnh lỗi trong mẫu": len(corrupted),
    "Số chiều HOG": dims["hog"],
    "Số chiều histogram màu": dims["color"],
    "Tổng số chiều đặc trưng (trước PCA)": dims["total"],
    f"Số chiều để giữ 95% phương sai (mẫu {X.shape[0]} ảnh)": ks["Giữ 95%"],
    f"% phương sai giữ lại với {k_default} component": round(cum[k_default - 1], 1),
}, name="Giá trị")
summary.to_frame()

**Kết luận / quyết định preprocessing** *(điền sau khi chạy notebook trên dataset thật)*:

- Có class imbalance không? Có cần `class_weight` không? ...
- Kích thước ảnh phổ biến: ... → resize về 128×128 cho HOG+SVM, 224×224 cho CNN/MobileNetV2 vì: ...
- Có ảnh lỗi/bất thường không? ...
- Chọn `n_components` = ... cho PCA vì giữ được ...% phương sai mà giảm ...% số chiều.
- Cần augmentation mạnh không (vì sao)? ...